In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import nltk
import spacy

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

import shap
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

ModuleNotFoundError: No module named 'spacy'

### Load Dataset

In [3]:
df=pd.read_csv("data/your_dataset_5000.csv")

In [4]:
df.shape

(5000, 2)

In [5]:
df.dropna()

,text,label
0,Exercise plays a crucial role in supporting me...,1
1,Renewable energy helps fight climate change by...,1
2,"A futuristic smart city is a vibrant, intercon...",1
3,Healthy eating habits are especially important...,1
4,Machine learning is transforming healthcare by...,1
...,...,...
4995,Machine learning models can now detect cancer ...,1
4996,Blockchain can provide secure and transparent ...,1
4997,Volunteering at the animal shelter was a rewar...,0
4998,Machine learning models can now detect cancer ...,1


In [7]:
df.shape

(5000, 2)

In [10]:
df.isnull().sum()

text     0
label    0
dtype: int64

In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    5000 non-null   object
 1   label   5000 non-null   int64 
dtypes: int64(1), object(1)
memory usage: 78.3+ KB


In [12]:
df=df.sample(frac=1,random_state=42).reset_index(drop=True)

In [13]:
print(len(df[df['label']==0]))

2500


In [14]:
print(len(df[df['label']==1]))

2500


### Feature Extraction

In [ ]:
from nltk.corpus import stopwords
stop_words = set(stopwords.words('english'))

def lexical_features(text):
    words = word_tokenize(text.lower())
    words_only = [w for w in words if w.isalpha()]
    
    total_words = len(words_only)
    unique_words = len(set(words_only))
    
    # Type Token Ratio — how varied is the vocabulary?
    ttr = unique_words / total_words if total_words > 0 else 0
    
    # Average word length
    avg_word_len = sum(len(w) for w in words_only) / total_words if total_words > 0 else 0
    
    # How many filler/transition words AI loves to use
    ai_words = ["furthermore", "moreover", "additionally", "however", 
                "therefore", "consequently", "nevertheless", "thus"]
    ai_word_count = sum(1 for w in words_only if w in ai_words)
    ai_word_ratio = ai_word_count / total_words if total_words > 0 else 0
    
    return {
        "ttr":           ttr,
        "avg_word_len":  avg_word_len,
        "ai_word_ratio": ai_word_ratio,
        "total_words":   total_words
    }

# Test it
print(lexical_features("I went to the market today and honestly it was a mess."))

In [ ]:
def syntactic_features(text):
    sentences = sent_tokenize(text)
    num_sentences = len(sentences)
    
    if num_sentences == 0:
        return {"avg_sent_len": 0, "sent_len_variance": 0, "num_sentences": 0}
    
    sent_lengths = [len(word_tokenize(s)) for s in sentences]
    
    # Average sentence length
    avg_sent_len = sum(sent_lengths) / num_sentences
    
    # Variance — humans have HIGH variance, AI has LOW variance
    sent_len_variance = pd.Series(sent_lengths).var() if num_sentences > 1 else 0
    
    return {
        "avg_sent_len":      avg_sent_len,
        "sent_len_variance": sent_len_variance,
        "num_sentences":     num_sentences
    }

# Test it
print(syntactic_features("I went to market. It was crowded and noisy and I hated it. Fine."))

In [ ]:
def stylometric_features(text):
    sentences = sent_tokenize(text)
    sent_lengths = [len(word_tokenize(s)) for s in sentences]
    
    # Burstiness — key feature!
    # High burstiness = human (short + long sentences mixed)
    # Low burstiness  = AI   (all sentences similar length)
    if len(sent_lengths) > 1:
        mean = pd.Series(sent_lengths).mean()
        std  = pd.Series(sent_lengths).std()
        burstiness = std / mean if mean > 0 else 0
    else:
        burstiness = 0
    
    words = word_tokenize(text.lower())
    
    # Hedging words — humans use these casually
    hedge_words = ["maybe", "perhaps", "probably", "i think", 
                   "i guess", "kind of", "sort of", "not sure"]
    hedge_count = sum(1 for w in words if w in hedge_words)
    
    # Passive voice indicator words
    passive_words = ["was", "were", "been", "being", "is", "are"]
    passive_count = sum(1 for w in words if w in passive_words)
    passive_ratio = passive_count / len(words) if len(words) > 0 else 0
    
    return {
        "burstiness":    burstiness,
        "hedge_count":   hedge_count,
        "passive_ratio": passive_ratio
    }

# Test it
print(stylometric_features("Maybe I was wrong. I think it could have gone differently. Not sure though."))

In [ ]:
def punctuation_features(text):
    total_chars = len(text)
    
    comma_ratio     = text.count(",")  / total_chars if total_chars > 0 else 0
    exclaim_ratio   = text.count("!")  / total_chars if total_chars > 0 else 0
    question_ratio  = text.count("?")  / total_chars if total_chars > 0 else 0
    ellipsis_count  = text.count("...")
    
    return {
        "comma_ratio":    comma_ratio,
        "exclaim_ratio":  exclaim_ratio,
        "question_ratio": question_ratio,
        "ellipsis_count": ellipsis_count
    }

# Test it
print(punctuation_features("Wait... really? That's amazing, wow!! I can't believe it."))

In [ ]:
# This combines ALL features into one row
def extract_features(text):
    features = {}
    features.update(lexical_features(text))
    features.update(syntactic_features(text))
    features.update(stylometric_features(text))
    features.update(punctuation_features(text))
    return features

# Test on one text
print(extract_features("I went to the market today. It was okay I guess."))

In [ ]:
print("Extracting features... please wait")

# Apply to every row in df
feature_df = df["text"].apply(extract_features)
feature_df = pd.DataFrame(feature_df.tolist())

print(f"Done! Feature matrix shape: {feature_df.shape}")
feature_df.head()